# 17 — Transactions, Constraints, and Exactly-Once Semantics

Atomicity matters more than application-level checks under concurrency. Practice schema-enforced uniqueness, rollback, idempotency records, and failure-safe multi-step updates.

*Advanced · 90–150 min*

## You will learn
- Enforce constraints in a real database rather than only memory
- Use SQLite transactions correctly
- Design a persistent idempotency key table
- Recognize the difference between at-least-once delivery and exactly-once effects

## Examples

### 1. Atomic transactions

A multi-step transfer should either update both accounts or neither. Use a transaction context manager and a CHECK constraint; if any statement fails, the transaction rolls back. Close the connection explicitly.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 20)])
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance = balance - 150 WHERE id = 1")
            db.execute("UPDATE accounts SET balance = balance + 150 WHERE id = 2")
    except sqlite3.IntegrityError:
        print("Transfer rolled back")
    print(db.execute("SELECT id, balance FROM accounts ORDER BY id").fetchall())
finally:
    db.close()

**Review:** What happens if the sender has insufficient funds during a real transfer?

### 2. Unique constraints beat check-then-insert

Checking for duplicates before insertion is not atomic under concurrency. A UNIQUE constraint can enforce one active identifier at storage level, and conflicts can be handled deliberately.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE jobs (job_id TEXT PRIMARY KEY, status TEXT NOT NULL)")
    db.execute("INSERT INTO jobs VALUES (?, ?)", ("J1", "queued"))
    try:
        db.execute("INSERT INTO jobs VALUES (?, ?)", ("J1", "queued"))
    except sqlite3.IntegrityError:
        print("Duplicate job rejected")
finally:
    db.close()

**Review:** Why might an in-memory set of seen job IDs fail in a multi-instance service?

### 3. Idempotency in persistent storage

A durable idempotency table can associate a logical operation key with a stored result. The key and side effect should be committed atomically; otherwise retrying after a crash can duplicate the effect.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE outcomes (request_key TEXT PRIMARY KEY, result TEXT NOT NULL)")
    def process(key, outcome):
        with db:
            db.execute("INSERT OR IGNORE INTO outcomes VALUES (?, ?)", (key, outcome))
            return db.execute("SELECT result FROM outcomes WHERE request_key=?", (key,)).fetchone()[0]
    print(process("order-1", "confirmed"))
    print(process("order-1", "different"))
finally:
    db.close()

**Review:** What validation is needed if the same key is reused with a different request body?

### 4. Resource cleanup and reliability

The sqlite3 connection context manager commits or rolls back but does not close the connection. Keep ownership explicit, and use connection.close in finally or contextlib.closing. Do not treat network calls inside database transactions casually.

In [ ]:
from contextlib import closing
import sqlite3

with closing(sqlite3.connect(":memory:")) as connection:
    with connection:
        connection.execute("CREATE TABLE events (id INTEGER PRIMARY KEY, name TEXT)")
        connection.execute("INSERT INTO events (name) VALUES (?)", ("ready",))
    count = connection.execute("SELECT COUNT(*) FROM events").fetchone()[0]
print("Rows:", count)

**Review:** Why should a transaction avoid waiting on slow external HTTP requests?

## Exercises

**Exercise 1.** Implement a transactional two-account transfer with all invariants enforced.

In [ ]:
# Exercise 1
# TODO: implement and test.

**Exercise 2.** Design an idempotency table rejecting conflicting payloads for one key.

In [ ]:
# Exercise 2
# TODO: implement and test.

**Exercise 3.** Write a test showing rollback after a second statement fails.

In [ ]:
# Exercise 3
# TODO: implement and test.

**Exercise 4.** Explain what exactly-once effects mean in a system with retried delivery.

In [ ]:
# Exercise 4
# TODO: implement and test.

## Common mistakes

- check-then-insert alone can race under concurrent callers.
- A transaction context does not automatically close a SQLite connection.
- Exactly-once effects require carefully coordinated durable state and duplicate handling.

**Next:** [18 — Production Quality: Packaging, CI, Configuration, and Deployment](18_production_quality_ci_deployment.ipynb)